# 🎬 User-Based Collaborative Filtering

## 1. Introduction

### What is Collaborative Filtering?

Collaborative Filtering is a recommendation technique that suggests items to users based on the preferences and behaviour of other users.

It works on the idea that users who have shown similar preferences in the past may also have similar preferences in the future.

### What is User-Based Collaborative Filtering?

User-Based Collaborative Filtering finds users with similar rating patterns and uses their preferences to recommend movies.

**Example:**

- User A and User B have rated several movies similarly.
- User B has watched and highly rated a movie that User A hasn't watched.
- We can recommend that movie to User A.

### Our Objective

Build a movie recommendation system that:

1. Identifies users with similar movie preferences.
2. Calculates similarity between users.
3. Finds movies liked by similar users.
4. Recommends movies that the target user hasn't rated.

### Dataset

We will use the **MovieLens Latest Small Dataset** provided by GroupLens Research.

- 610 users
- 9,742 movies
- 100,836 ratings
- Ratings range from 0.5 to 5.0 stars.

**Dataset source:** https://grouplens.org/datasets/movielens/latest/

### Implementation Workflow

`Data Loading → Data Exploration → User-Movie Matrix → Similarity Calculation → Recommendations → Evaluation`

In [1]:
import pandas as pd #used to load, manipulate, and analyze data
import numpy as np #used for numerical computations

ratings = pd.read_csv("../data/ml-latest-small/ratings.csv")

ratings.head()


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


##  Understanding Timestamps

The `timestamp` column represents the time at which a user submitted a movie rating.

MovieLens stores timestamps in **Unix format**.

### What is a Unix Timestamp?

A Unix timestamp represents the number of seconds elapsed since January 1, 1970 (UTC).

Since these values are not human-readable, we can convert them into standard date and time format using Pandas.

### Conversion

We use `pd.to_datetime()` with `unit="s"` to convert Unix timestamps into readable dates.

In [2]:
pd.to_datetime(ratings["timestamp"], unit="s")
# Convert Unix timestamps into readable datetime format

0        2000-07-30 18:45:03
1        2000-07-30 18:20:47
2        2000-07-30 18:37:04
3        2000-07-30 19:03:35
4        2000-07-30 18:48:51
                 ...        
100831   2017-05-03 21:53:22
100832   2017-05-03 22:21:31
100833   2017-05-08 19:50:47
100834   2017-05-03 21:19:12
100835   2017-05-03 21:20:15
Name: timestamp, Length: 100836, dtype: datetime64[s]

Data Exploration

Before building our recommendation system, we need to understand the structure and quality of our dataset.

We will examine:

1. Number of rows and columns.
2. Number of unique users.
3. Number of unique movies.
4. Missing values in each column.

This will help us understand the data available for building our recommendation system.

In [3]:
#checks the dimensions of the dataset
print("Dataset shape:",ratings.shape)

#count the number of unique users 
print("unique users: ", ratings["userId"].nunique())

#count the number of unique movies
print("unique movies : ", ratings["movieId"].nunique())

#check for missing values
print("missing values:\n", ratings.isnull().sum())


Dataset shape: (100836, 4)
unique users:  610
unique movies :  9724
missing values:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


### Observations

After exploring the dataset, we found:

- **100,836 rows:** Each row represents one user-movie rating.
- **4 columns:** userId, movieId, rating and timestamp.
- **610 unique users:** The dataset contains ratings from 610 users.
- **9,724 unique movies:** A total of 9,724 movies have received at least one rating.
- **No missing values:** All four columns are complete.

The dataset is ready for further processing.

**Key takeaway:** We have individual user ratings, which allows us to study similarities between users and build a Collaborative Filtering recommendation system.

Creating the User-Movie Rating Matrix

Collaborative Filtering requires us to organise ratings into a matrix.

We will use a **pivot table** to transform our dataset:

- Rows → Users (`userId`)
- Columns → Movies (`movieId`)
- Values → Ratings (`rating`)

Each cell represents the rating given by a particular user to a particular movie.

If a user hasn't rated a movie, the cell will contain `NaN`.

This matrix will help us compare users based on the movies they have both rated.

In [4]:
# Create the User-Movie Rating Matrix
user_movie_matrix = ratings.pivot(
    index = "userId",
    columns = "movieId",
    values = "rating"
)

In [5]:
user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
user_movie_matrix.shape

(610, 9724)

### Understanding the Matrix Dimensions

The resulting User-Movie Matrix has a shape of **(610, 9724)**.

- 610 rows represent the unique users.
- 9,724 columns represent the unique movies.
- Each cell stores the corresponding user's rating.
- Missing ratings are represented by `NaN`.

This matrix allows us to compare users based on their ratings for common movies.

Understanding Matrix Sparsity

Our User-Movie Matrix contains many missing values because users have rated only a small portion of all available movies.

This is known as **data sparsity**.

We will calculate the percentage of missing values to understand how sparse our matrix is.

In [7]:
#calculate the percentage of missing ratings 
missing_percentage = user_movie_matrix.isnull().sum().sum() / user_movie_matrix.size * 100
print(f"missing values : {missing_percentage:.2f}%")

missing values : 98.30%


### Observations: Matrix Sparsity

Our User-Movie Matrix contains approximately **98.30% missing values**.

This indicates high data sparsity.

**Why does this happen?**

- Each user watches and rates only a small subset of all available movies.
- Most users have not rated the majority of movies in the dataset.

**Impact on our recommendation system:**

- Users may have very few commonly rated movies.
- Similarity calculations can become less reliable when users have limited overlapping ratings.
- Finding sufficiently similar users becomes challenging.

Handling data sparsity is an important challenge in Collaborative Filtering.

### Why Are Common Ratings Important?

Similarity between two users should be calculated using movies rated by both users.

However, having very few common movies can produce unreliable similarity scores.

For example, two users may give the same rating to one movie but have completely different preferences otherwise.

Therefore, we should consider the number of common ratings when identifying similar users.

This helps reduce unreliable recommendations caused by limited overlapping data.

## Calculating User Similarity

To recommend movies based on similar users, we first need to measure how similar their rating patterns are.

We will use **Pearson Correlation Coefficient**.

### What is Pearson Correlation?

Pearson Correlation measures the strength and direction of the linear relationship between two sets of values.

In our project, it measures how similarly two users rate the movies they have both watched.

### Interpretation

| Correlation | Meaning |
|---|---|
| +1 | Perfectly similar rating patterns |
| 0 | No linear correlation |
| -1 | Opposite rating patterns |

### Why Pearson Correlation?

Different users may have different rating habits.

For example, one user might generally give movies 4–5 stars, while another usually gives 2–3 stars.

Pearson Correlation focuses on how their ratings vary relative to their own averages, rather than simply comparing absolute rating values.

### Pearson Correlation Formula

The Pearson Correlation Coefficient is calculated as:

r = Σ[(xᵢ - x̄)(yᵢ - ȳ)] / √[Σ(xᵢ - x̄)² × Σ(yᵢ - ȳ)²]

Where:

- xᵢ = Rating given by User X
- yᵢ = Rating given by User Y
- x̄ = Average rating of User X
- ȳ = Average rating of User Y

The formula compares how each user's ratings differ from their own average ratings.

We calculate it using only movies rated by both users.

### Calculating User Similarity

We will calculate the Pearson correlation between users using their common movie ratings.

Pandas provides the `.corrwith()` function to calculate correlations between rows or columns.

Since users are represented as rows, we will transpose our User-Movie Matrix so that users become columns.

This allows us to calculate correlations between users.

Missing ratings will be handled by Pandas, which excludes non-overlapping values when calculating pairwise correlations.

In [8]:
#calculate pearson correlation btw users
user_similarity = user_movie_matrix.T.corr(method = "pearson")

#display similarity matrix
user_similarity.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,NaN,0.079819,0.207983,0.268749,-0.291636,-0.118773,0.469668,0.918559,-0.037987,...,9.157371e-02,-1.597727e-16,-0.061503,-0.407556,-0.164871,0.066378,0.174557,0.268070,-0.175412,-0.032086
2,NaN,1.0,NaN,NaN,NaN,NaN,-0.991241,NaN,NaN,0.037796,...,-3.873468e-01,NaN,-1.000000,NaN,NaN,0.583333,NaN,-0.125000,NaN,0.623288
3,0.079819,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.433200,NaN,NaN,-0.791334,-0.333333,-0.395092,NaN,0.569562
4,0.207983,NaN,NaN,1.000000,-0.336525,0.148498,0.542861,0.117851,NaN,0.485794,...,-2.221127e-01,3.966413e-01,0.090090,-0.080296,0.400124,0.144603,0.116518,-0.170501,-0.277350,-0.043786
5,0.268749,NaN,NaN,-0.336525,1.000000,0.043166,0.158114,0.028347,NaN,-0.777714,...,2.719480e-16,1.533034e-01,0.234743,0.067791,-0.364156,0.244321,0.231080,-0.020546,0.384111,0.040582


In [9]:
user_similarity.shape

(610, 610)

## Calculating Common Ratings Between Users

Before selecting similar users, we need to calculate how many movies each pair of users has rated in common.

Why is this important?

- Two users may have a high Pearson correlation based on very few movies.
- Similarity based on more common ratings is generally more reliable.
- We will use a minimum common-rating threshold to filter unreliable similarities.

We will first create a matrix showing whether a user has rated each movie, and then multiply this matrix by its transpose to count common ratings between every pair of users.

In [10]:
#create a binary matrix
# 1 means the user rated the movie
# 0 means the user did not rate the movie

rating_presence = user_movie_matrix.notna().astype(int)

#count the number of commonly rated movies
# btw every pair of users

common_ratings = rating_presence @ rating_presence.T

#display result
common_ratings.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,232,2,7,45,13,33,26,15,5,6,...,11,27,110,12,35,84,60,136,9,69
2,2,29,0,1,1,2,3,1,0,5,...,11,1,2,0,0,5,1,6,1,18
3,7,0,39,1,1,3,0,1,0,0,...,2,2,9,0,1,8,4,9,0,7
4,45,1,1,216,12,27,22,9,1,9,...,15,23,139,9,16,96,29,74,3,53
5,13,1,1,12,44,36,9,19,0,3,...,4,31,25,19,14,23,15,30,10,14


In [11]:
common_ratings.shape

(610, 610)

## 10. Filtering User Similarities

We will now filter our Pearson correlation matrix using a minimum common-rating threshold of 5.

Only user pairs who have rated at least 5 movies in common will retain their similarity score.

Pairs with fewer than 5 common ratings will be assigned NaN because their similarity score may not be sufficiently reliable.

**Note:** A threshold of 5 is our initial experimental choice, not a universal rule. We can compare different thresholds during model evaluation.

In [12]:
#set the min num of common ratings
min_common_ratings = 5

#keep similarities only when users have enough common ratings
user_similarity_filtered = user_similarity.where(
    common_ratings >= min_common_ratings
)

#display filtered similarity matrix
user_similarity_filtered.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,NaN,0.079819,0.207983,0.268749,-0.291636,-0.118773,0.469668,0.918559,-0.037987,...,0.091574,-1.597727e-16,-0.061503,-0.407556,-0.164871,0.066378,0.174557,0.268070,-0.175412,-0.032086
2,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.037796,...,-0.387347,NaN,NaN,NaN,NaN,0.583333,NaN,-0.125000,NaN,0.623288
3,0.079819,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.433200,NaN,NaN,-0.791334,NaN,-0.395092,NaN,0.569562
4,0.207983,NaN,NaN,1.000000,-0.336525,0.148498,0.542861,0.117851,NaN,0.485794,...,-0.222113,3.966413e-01,0.090090,-0.080296,0.400124,0.144603,0.116518,-0.170501,NaN,-0.043786
5,0.268749,NaN,NaN,-0.336525,1.000000,0.043166,0.158114,0.028347,NaN,NaN,...,NaN,1.533034e-01,0.234743,0.067791,-0.364156,0.244321,0.231080,-0.020546,0.384111,0.040582


In [13]:
user_similarity_filtered.shape

(610, 610)

## Finding Similar Users (Neighbours)

In User-Based Collaborative Filtering, we recommend movies to a target user by identifying other users with similar rating patterns.

These similar users are called **neighbours**.

For our initial implementation:
- We will select the top 10 most similar users.
- We will consider only positive similarity scores.
- We will exclude the target user themselves.

These neighbours will help us identify movies that the target user has not watched but that similar users have rated.

In [14]:
#select a target user
target_user = 1

#get similarity scores of the target user with all other users
similar_users = user_similarity_filtered[target_user]

#remove the target user and keep only pos similarities
similar_users = similar_users.drop(target_user)
similar_users = similar_users[similar_users > 0]

#select the 10 most similar users
top_neighbours = similar_users.sort_values(ascending= False).head(10)

top_neighbours


userId
511    0.925820
9      0.918559
13     0.878310
366    0.872872
401    0.866921
535    0.866400
90     0.821584
157    0.801784
139    0.790569
476    0.786936
Name: 1, dtype: float64

##  Identifying Unrated Movies

After finding the top 10 neighbours, we need to identify movies that they have rated but our target user has not.

These movies become potential recommendations.

We will first extract the ratings given by the selected neighbours.

In [15]:
#get ratings given by the top neighbours
neighbour_ratings = user_movie_matrix.loc[top_neighbours.index]

#display their ratings
neighbour_ratings.head(10)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
511,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
366,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
401,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
535,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,3.0,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
139,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Filtering Unrated Movies

We should not recommend movies that the target user has already rated.

Therefore, we will identify all movies with missing ratings (`NaN`) in the target user's row.

These movies will be our recommendation candidates.

In [16]:
#get all movie ratings of the target user
target_user_ratings = user_movie_matrix.loc[target_user]

#identify movies that the target user has not rated
unrated_movies = target_user_ratings[target_user_ratings.isna()].index

print(len(unrated_movies))

9492


##  Predicting Movie Ratings

We will predict the target user's rating for each unrated movie using a similarity-weighted average of ratings given by their neighbours.

A neighbour with a higher similarity score will have more influence on the predicted rating.

We will only use neighbours who have actually rated the movie.

The denominator will therefore contain the sum of absolute similarity scores of the neighbours who rated that particular movie.

In [17]:
#get similarity scores of the selected neighbours
neighbour_similarities = top_neighbours

#calculate weighted ratings
weighted_ratings = neighbour_ratings.mul(neighbour_similarities, axis = 0)

#sum weighted ratings for each movie
sum_weighted_ratings = weighted_ratings.sum(axis = 0)

#sum absolute similarity scores only for neighbours who rated each movie 
sum_similarities =neighbour_ratings.notna().mul(
    neighbour_similarities.abs(), axis=0
).sum(axis=0)

# Calculate predicted ratings
predicted_ratings = sum_weighted_ratings / sum_similarities

# Keep only movies the target user has not rated
predicted_ratings = predicted_ratings.loc[unrated_movies]

# Remove movies that cannot receive a prediction
predicted_ratings = predicted_ratings.dropna()

predicted_ratings.head()

movieId
2     4.0
7     4.0
10    3.0
11    3.0
13    3.0
dtype: float64

Movie 2 is predicted to receive a rating of 4.0 from User 1.
Movie 7 is also predicted to receive 4.0.
Movie 10 is predicted to receive 3.0.
Important: These are estimated ratings, not actual ratings.

## Generating Top Recommendations

After predicting ratings for unrated movies, we can rank them in descending order.

Movies with higher predicted ratings will appear first.

We will select the top 10 movies as recommendations for our target user.

In [18]:
# Sort predicted ratings in descending order
top_recommendations = predicted_ratings.sort_values(
    ascending=False
).head(10)

top_recommendations

movieId
1057    5.0
1041    5.0
52      5.0
14      5.0
25      5.0
58      5.0
82      5.0
984     5.0
613     5.0
1034    5.0
dtype: float64

## Checking Recommendation Support

A predicted rating alone does not tell us how much evidence supports a recommendation.

For example, a predicted rating of 5.0 based on one neighbour is less reliable than a rating of 5.0 supported by eight neighbours.

We will count how many of our selected neighbours rated each recommended movie.

In [19]:
# Count how many neighbours rated each movie
neighbour_rating_counts = neighbour_ratings.notna().sum(axis=0)

# Get the number of ratings supporting each top recommendation
recommendation_support = neighbour_rating_counts.loc[
    top_recommendations.index
]

# Combine predicted ratings and neighbour counts
recommendation_details = pd.DataFrame({
    "predicted_rating": top_recommendations,
    "neighbour_count": recommendation_support
})

recommendation_details

,predicted_rating,neighbour_count
movieId,,
1057,5.0,1
1041,5.0,1
52,5.0,1
14,5.0,1
25,5.0,1
58,5.0,1
82,5.0,1
984,5.0,1
613,5.0,1


## Filtering Recommendations by Neighbour Support

A recommendation supported by multiple similar users is generally more reliable than one supported by only a single user.

Therefore, we will introduce a minimum neighbour-support condition.

For our initial experiment, a movie must have ratings from at least 2 of the selected 10 neighbours to qualify for recommendation.

This helps reduce recommendations based on very limited evidence.


In [20]:
# Count how many neighbours rated each movie
neighbour_rating_counts = neighbour_ratings.notna().sum(axis=0)

# Combine predictions and neighbour support for all movies
all_recommendations = pd.DataFrame({
    "predicted_rating": predicted_ratings,
    "neighbour_count": neighbour_rating_counts
})

# Require support from at least 2 neighbours
min_neighbour_support = 2

reliable_recommendations = all_recommendations[
    all_recommendations["neighbour_count"] >= min_neighbour_support
]

# Sort and select top 10
reliable_recommendations = reliable_recommendations.sort_values(
    by="predicted_rating",
    ascending=False
).head(10)

reliable_recommendations

,predicted_rating,neighbour_count
movieId,,
318,5.000000,2
109487,4.742641,2
589,4.474113,2
91529,4.357562,3
4993,4.323195,5
7153,4.143318,4
5952,4.020045,3
32,4.000000,2
3753,4.000000,2


## Mapping Movie IDs to Movie Titles

Our recommendation model currently returns MovieLens movie IDs.

To make the recommendations understandable, we need to map these IDs to their corresponding movie titles and genres.

We will use the MovieLens `movies.csv` dataset and merge it with our recommendation results using `movieId` as the common column.

In [21]:
# Load movie titles and genres
movies = pd.read_csv("../data/ml-latest-small/movies.csv")

# Display the first few rows
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [22]:
# Convert movieId index into a regular column
recommendation_results = reliable_recommendations.reset_index()

# Merge recommendations with movie details
recommendation_results = recommendation_results.merge(
    movies,
    on="movieId",
    how="left"
)

# Display relevant columns
recommendation_results[
    ["movieId", "title", "genres", "predicted_rating", "neighbour_count"]
]

,movieId,title,genres,predicted_rating,neighbour_count
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,5.000000,2
1,109487,Interstellar (2014),Sci-Fi|IMAX,4.742641,2
2,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,4.474113,2
3,91529,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX,4.357562,3
4,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,4.323195,5
5,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,4.143318,4
6,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,4.020045,3
7,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,4.000000,2
8,3753,"Patriot, The (2000)",Action|Drama|War,4.000000,2
9,595,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,3.951637,2


### Observations: First User-Based Recommendations

Our User-Based Collaborative Filtering model successfully generated its first 10 recommendations for User 1.

**Key observations:**

- The model identified similar users using Pearson correlation.
- The top 10 neighbours were selected based on positive similarity scores.
- Predicted ratings were calculated using similarity-weighted averages.
- Movies already rated by User 1 were excluded.
- A minimum neighbour-support condition of 2 was applied.
- Movie IDs were mapped to their corresponding titles and genres.

**Notable result:** The Shawshank Redemption received the highest predicted rating of 5.0, supported by 2 neighbours.

**Limitation:** These recommendations are preliminary. We still need to evaluate prediction accuracy, recommendation quality, coverage and the effect of different thresholds.

## Model Evaluation: Train-Validation-Test Split

To evaluate our recommendation system fairly, we divide the ratings dataset into three parts:

- **Training set (60%)**: Used to build the user-movie matrix and calculate user similarities.
- **Validation set (20%)**: Used to experiment with different model configurations and select the best one.
- **Testing set (20%)**: Used only for the final evaluation of the selected model.

We split the data separately for each user so that their ratings are distributed across all three sets.

This separation prevents us from using the test data while tuning the model, giving us a more reliable evaluation.

In [23]:
from sklearn.model_selection import train_test_split

## Model Evaluation: Train-Validation-Test Split

To evaluate our recommendation system fairly, we divide the ratings dataset into three parts:

- **Training set (60%)**: Used to build the user-movie matrix and calculate user similarities.
- **Validation set (20%)**: Used to experiment with different model configurations and select the best hyperparameters.
- **Testing set (20%)**: Used only for the final evaluation of the selected model.

We split the data separately for each user so that their ratings are distributed across all three sets.

This separation prevents us from using test data during model tuning, giving us a more reliable estimate of the model's performance.

In [24]:
train_data = []
validation_data = []
test_data = []

# Split ratings separately for every user
for user_id, user_ratings in ratings.groupby("userId"):

    # First separate 20% for testing
    train_validation, user_test = train_test_split(
        user_ratings,
        test_size=0.2,
        random_state=42
    )

    # Split the remaining 80% into training and validation
    user_train, user_validation = train_test_split(
        train_validation,
        test_size=0.25,
        random_state=42
    )

    train_data.append(user_train)
    validation_data.append(user_validation)
    test_data.append(user_test)

# Combine individual user splits
train_ratings = pd.concat(train_data)
validation_ratings = pd.concat(validation_data)
test_ratings = pd.concat(test_data)

print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))
print("Testing ratings:", len(test_ratings))

Training ratings: 60135
Validation ratings: 20284
Testing ratings: 20417


## Build the Model and Tune Hyperparameters

We use only the training data to build our recommendation model.

The process involves:

1. Creating the user-movie rating matrix.
2. Calculating Pearson similarity between users.
3. Filtering similarities based on the minimum number of commonly rated movies.
4. Creating a function to predict a user's rating for a movie.
5. Testing different numbers of neighbours using the validation dataset.

We compare the configurations using **MAE, RMSE, and prediction coverage** to select the best number of neighbours.

In [25]:
# ============================================
# 1. BUILD USER-MOVIE MATRIX
# ============================================

train_user_movie_matrix = train_ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)

# ============================================
# 2. CALCULATE USER SIMILARITIES
# ============================================

train_user_similarity = train_user_movie_matrix.T.corr(
    method="pearson"
)

# ============================================
# 3. COUNT COMMONLY RATED MOVIES
# ============================================

train_rating_presence = train_user_movie_matrix.notna().astype(int)

train_common_ratings = train_rating_presence @ train_rating_presence.T

# Apply minimum common ratings threshold
min_common_ratings = 3

train_user_similarity_filtered = train_user_similarity.where(
    train_common_ratings >= min_common_ratings
)

# ============================================
# 4. CREATE PREDICTION FUNCTION
# ============================================

def predict_rating(user_id, movie_id, n_neighbours):

    # Check whether the movie exists in training data
    if movie_id not in train_user_movie_matrix.columns:
        return np.nan

    # Get similarities of the target user with other users
    similarities = train_user_similarity_filtered.loc[user_id]

    # Exclude the target user
    similarities = similarities.drop(user_id)

    # Keep only positive similarities
    similarities = similarities[similarities > 0]

    # Select the top N neighbours
    neighbours = similarities.sort_values(
        ascending=False
    ).head(n_neighbours)

    # Get ratings given by neighbours for this movie
    neighbour_ratings = train_user_movie_matrix[movie_id].reindex(
        neighbours.index
    ).dropna()

    # Keep similarities of neighbours who rated this movie
    relevant_similarities = neighbours.reindex(
        neighbour_ratings.index
    )

    # Require at least 2 supporting neighbours
    if len(neighbour_ratings) < 2:
        return np.nan

    # Calculate similarity-weighted predicted rating
    prediction = (
        (relevant_similarities * neighbour_ratings).sum()
        / relevant_similarities.abs().sum()
    )

    return prediction


# ============================================
# 5. TEST DIFFERENT NEIGHBOURHOOD SIZES
# ============================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

neighbour_options = [10, 20, 30, 50, 100, 150, 200, 300]

validation_results = []

for n in neighbour_options:

    validation_predictions = validation_ratings.copy()

    validation_predictions["predicted_rating"] = [
        predict_rating(row.userId, row.movieId, n)
        for row in validation_predictions.itertuples()
    ]

    valid = validation_predictions.dropna(
        subset=["predicted_rating"]
    )

    mae = mean_absolute_error(
        valid["rating"],
        valid["predicted_rating"]
    )

    rmse = np.sqrt(mean_squared_error(
        valid["rating"],
        valid["predicted_rating"]
    ))

    coverage = len(valid) / len(validation_predictions) * 100

    validation_results.append({
        "Neighbours": n,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "Coverage": round(coverage, 2),
        "Successful_Predictions": len(valid)
    })

validation_results = pd.DataFrame(validation_results)

validation_results

,Neighbours,MAE,RMSE,Coverage,Successful_Predictions
0,10,0.8669,1.1217,8.42,1708
1,20,0.8428,1.0868,16.96,3440
2,30,0.8312,1.0760,23.96,4860
3,50,0.8073,1.0494,36.43,7389
4,100,0.7744,1.0026,61.27,12428
5,150,0.7527,0.9733,77.24,15667
6,200,0.7489,0.9686,83.87,17013
7,300,0.7434,0.9618,88.02,17853


## Final Evaluation on Test Data

We selected **300 neighbours** based on the validation results.

Now, we evaluate the selected configuration on the untouched test dataset. This gives us an unbiased estimate of how well our model performs on unseen ratings.

We use three evaluation metrics:

- **MAE (Mean Absolute Error):** Average absolute difference between actual and predicted ratings.
- **RMSE (Root Mean Squared Error):** Measures prediction error while penalising larger errors more heavily.
- **Coverage:** Percentage of test ratings for which the model successfully generates a prediction.

The test dataset is used only once for this final evaluation, not for hyperparameter tuning.

In [26]:
# ============================================
# FINAL TEST EVALUATION
# ============================================

best_neighbours = 300

# Generate predictions for test ratings
test_predictions = test_ratings.copy()

test_predictions["predicted_rating"] = [
    predict_rating(row.userId, row.movieId, best_neighbours)
    for row in test_predictions.itertuples()
]

# Keep successfully predicted ratings
valid_test_predictions = test_predictions.dropna(
    subset=["predicted_rating"]
)

# Calculate evaluation metrics
test_mae = mean_absolute_error(
    valid_test_predictions["rating"],
    valid_test_predictions["predicted_rating"]
)

test_rmse = np.sqrt(mean_squared_error(
    valid_test_predictions["rating"],
    valid_test_predictions["predicted_rating"]
))

test_coverage = (
    len(valid_test_predictions) / len(test_predictions)
) * 100

# Display final results
print("Selected neighbours:", best_neighbours)
print("MAE:", round(test_mae, 4))
print("RMSE:", round(test_rmse, 4))
print("Total test ratings:", len(test_predictions))
print("Successful predictions:", len(valid_test_predictions))
print("Prediction coverage:", round(test_coverage, 2), "%")


# Save final evaluation results
final_test_results = pd.DataFrame([{
    "Algorithm": "User-Based Collaborative Filtering",
    "Neighbours": best_neighbours,
    "Min_Common_Ratings": 3,
    "MAE": round(test_mae, 4),
    "RMSE": round(test_rmse, 4),
    "Coverage": round(test_coverage, 2)
}])

final_test_results.to_csv(
    "../results/user_based_cf_final_evaluation.csv",
    index=False
)

print("\nFinal evaluation saved successfully!")

Selected neighbours: 300
MAE: 0.7494
RMSE: 0.9701
Total test ratings: 20417
Successful predictions: 17997
Prediction coverage: 88.15 %

Final evaluation saved successfully!


## Final Observations and Conclusion

### 1. Selected Hyperparameters

| Parameter | Selected Value |
|---|---|
| Similarity Metric | Pearson Correlation |
| Minimum Common Ratings | 3 |
| Number of Neighbours | 300 |
| Similarity Condition | Positive similarities only |
| Minimum Neighbour Support | 2 |
| Prediction Method | Similarity-Weighted Average |

### 2. Final Test Performance

| Evaluation Metric | Result |
|---|---:|
| MAE | 0.7494 |
| RMSE | 0.9701 |
| Prediction Coverage | 88.15% |
| Successful Predictions | 17,997 / 20,417 |

### 3. Conclusion

We implemented a memory-based **User-User Collaborative Filtering** system using the MovieLens dataset.

The system identifies users with similar rating patterns using Pearson correlation and predicts unknown movie ratings using a similarity-weighted average of neighbouring users' ratings.

Hyperparameter tuning was performed using the validation dataset, while the test dataset was reserved exclusively for final evaluation.

The selected configuration uses 300 neighbours and achieves an MAE of 0.7494, RMSE of 0.9701, and prediction coverage of 88.15%.

### 4. Limitations

- The system cannot predict ratings for movies absent from the training data.
- Users with insufficient similarity information may not receive predictions.
- Similarity calculations can become computationally expensive as the number of users grows.
- User-based recommendations may be affected by sparse rating data.
- Rating prediction accuracy does not necessarily reflect recommendation ranking quality.

## Saving the User-Based CF Model

We save the training user-movie matrix and user similarity matrix as pickle files.

These files allow us to reuse the recommendation system without recalculating user similarities every time.

In [27]:
import pickle
import os

# Ensure the models folder exists
os.makedirs("../models", exist_ok=True)

# Save user-movie matrix
with open("../models/user_movie_matrix.pkl", "wb") as file:
    pickle.dump(train_user_movie_matrix, file)

# Save user similarity matrix
with open("../models/user_similarity.pkl", "wb") as file:
    pickle.dump(train_user_similarity_filtered, file)

print("User-Based CF model artifacts saved successfully!")

User-Based CF model artifacts saved successfully!


### Load and Test the Saved Model

In [28]:
import sys
sys.path.append("../src")

from user_based_cf import UserBasedCF

# Load the saved matrices
with open("../models/user_movie_matrix.pkl", "rb") as file:
    saved_matrix = pickle.load(file)

with open("../models/user_similarity.pkl", "rb") as file:
    saved_similarity = pickle.load(file)

# Create recommender object
recommender = UserBasedCF(saved_matrix, saved_similarity)

# Generate recommendations for user 1
recommendations = recommender.recommend(
    user_id=1,
    n_neighbours=300,
    n_recommendations=10
)

recommendations

movieId
27815     5.000000
170705    5.000000
3074      5.000000
3296      5.000000
1238      5.000000
89904     5.000000
160271    5.000000
7121      4.939031
97866     4.934997
7842      4.933463
dtype: float64

### Display Movie Recommendations

We map the recommended movie IDs to their corresponding titles and genres using the MovieLens movie metadata.

This makes the recommendations easier to interpret than displaying movie IDs alone.

In [29]:
# Load MovieLens movie information
movies = pd.read_csv("../data/ml-latest-small/movies.csv")

# Convert recommendations into a DataFrame
recommendation_df = recommendations.rename(
    "predicted_rating"
).reset_index()

# Merge movie titles and genres
recommendation_df = recommendation_df.merge(
    movies,
    on="movieId",
    how="left"
)

recommendation_df[["movieId", "title", "genres", "predicted_rating"]]

,movieId,title,genres,predicted_rating
0,27815,"Chorus, The (Choristes, Les) (2004)",Drama,5.000000
1,170705,Band of Brothers (2001),Action|Drama|War,5.000000
2,3074,Jeremiah Johnson (1972),Western,5.000000
3,3296,To Sir with Love (1967),Drama,5.000000
4,1238,Local Hero (1983),Comedy,5.000000
5,89904,The Artist (2011),Comedy|Drama|Romance,5.000000
6,160271,Central Intelligence (2016),Action|Comedy,5.000000
7,7121,Adam's Rib (1949),Comedy|Romance,4.939031
8,97866,"Imposter, The (2012)",Documentary,4.934997
9,7842,Dune (2000),Drama|Fantasy|Sci-Fi,4.933463
